In [ ]:
#########################
# 0. 환경 설정
from google.colab import drive
drive.mount('/content/drive')

In [5]:
#########################
# 1. 가지치기
import torch
from torch.nn.utils import prune
from transformers import BertTokenizer, BertForSequenceClassification


tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    do_lower_case=False,
)
model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
)
#model.load_state_dict(torch.load("/content/drive/MyDrive/models/BertForSequenceClassification.pt"))
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/models/BertForSequenceClassification.pt",
        map_location=torch.device('cpu')
    )
)

print("가지치기 적용 전:")
print(model.bert.encoder.layer[0].attention.self.key.weight)

parameters = [
    (model.bert.embeddings.word_embeddings, "weight"),
    (model.bert.encoder.layer[0].attention.self.key, "weight"),
    (model.bert.encoder.layer[1].attention.self.key, "weight"),
    (model.bert.encoder.layer[2].attention.self.key, "weight"),
]
prune.global_unstructured(
    parameters=parameters,
    pruning_method=prune.L1Unstructured,
    amount=0.2
)

print("가지치기 적용 후:")
print(model.bert.encoder.layer[0].attention.self.key.weight)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


가지치기 적용 전:
Parameter containing:
tensor([[-0.0255, -0.0595, -0.0397,  ...,  0.0078, -0.0270, -0.0035],
        [ 0.0420,  0.0560,  0.0041,  ..., -0.0044,  0.0199,  0.0170],
        [ 0.0167, -0.0440, -0.0088,  ..., -0.0116, -0.0878, -0.0522],
        ...,
        [-0.0312, -0.0220,  0.0342,  ..., -0.0009,  0.0104,  0.0089],
        [-0.0753, -0.0749, -0.0068,  ..., -0.0132, -0.0329, -0.0990],
        [ 0.0163, -0.0421, -0.0056,  ...,  0.0444, -0.0184,  0.0413]],
       requires_grad=True)
가지치기 적용 후:
tensor([[-0.0255, -0.0595, -0.0397,  ...,  0.0000, -0.0270, -0.0000],
        [ 0.0420,  0.0560,  0.0000,  ..., -0.0000,  0.0199,  0.0170],
        [ 0.0167, -0.0440, -0.0000,  ..., -0.0116, -0.0878, -0.0522],
        ...,
        [-0.0312, -0.0220,  0.0342,  ..., -0.0000,  0.0000,  0.0000],
        [-0.0753, -0.0749, -0.0000,  ..., -0.0132, -0.0329, -0.0990],
        [ 0.0163, -0.0421, -0.0000,  ...,  0.0444, -0.0184,  0.0413]],
       grad_fn=<MulBackward0>)
